# Stage 1: Initial statistical spelling correction preprocessing

## Spelling corrector class

In [1]:
import re
import math
from collections import defaultdict, Counter
import numpy as np

class StatisticalSpellingCorrector:
    """
    Statistical Machine Translation approach to spelling correction.
    Treats spelling correction as translation from misspelled to correct English.
    """
    
    def __init__(self, n=2):
        """
        Initialize the Statistical spelling corrector.
        
        Args:
            n: n-gram size for language model (default: 2 for bigrams)
        """
        self.n = n
        # Translation model: P(misspelled|correct)
        self.translation_probs = defaultdict(lambda: defaultdict(float))
        # Language model: P(word|context)
        self.language_model = defaultdict(lambda: defaultdict(float))
        self.unigram_counts = Counter()
        self.vocabulary = set()
        
    def train_translation_model(self, parallel_corpus):
        """
        Train translation model from parallel corpus of (misspelled, correct) pairs.
        
        Args:
            parallel_corpus: List of tuples [(misspelled_sentence, correct_sentence), ...]
        """
        print("Training translation model...")
        
        # Count alignments
        alignment_counts = defaultdict(lambda: defaultdict(int))
        correct_word_counts = defaultdict(int)
        
        for misspelled_sent, correct_sent in parallel_corpus:
            misspelled_words = misspelled_sent.lower().split()
            correct_words = correct_sent.lower().split()
            
            # Simple word alignment (assumes similar word order)
            for m_word in misspelled_words:
                for c_word in correct_words:
                    # Increment alignment count based on edit distance
                    if self._should_align(m_word, c_word):
                        alignment_counts[c_word][m_word] += 1
                        correct_word_counts[c_word] += 1
                        self.vocabulary.add(c_word)
        
        # Calculate translation probabilities
        for c_word, m_words in alignment_counts.items():
            for m_word, count in m_words.items():
                # P(misspelled|correct)
                self.translation_probs[c_word][m_word] = count / correct_word_counts[c_word]
        
        print(f"Translation model trained with {len(self.vocabulary)} vocabulary words")
    
    def _should_align(self, word1, word2, threshold=3):
        """
        Determine if two words should be aligned based on edit distance.
        """
        if word1 == word2:
            return True
        
        edit_dist = self._edit_distance(word1, word2)
        # Align if edit distance is small relative to word length
        return edit_dist <= threshold
    
    def _edit_distance(self, s1, s2):
        """
        Calculate Levenshtein edit distance between two strings.
        """
        if len(s1) < len(s2):
            return self._edit_distance(s2, s1)
        
        if len(s2) == 0:
            return len(s1)
        
        previous_row = range(len(s2) + 1)
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                # Cost of insertions, deletions, or substitutions
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        
        return previous_row[-1]
    
    def train_language_model(self, corpus):
        """
        Train n-gram language model from correct English corpus.
        
        Args:
            corpus: List of correct English sentences
        """
        print(f"Training {self.n}-gram language model...")
        
        ngram_counts = defaultdict(lambda: defaultdict(int))
        context_counts = defaultdict(int)
        
        for sentence in corpus:
            words = ['<s>'] * (self.n - 1) + sentence.lower().split() + ['</s>']
            self.vocabulary.update(words)
            
            # Count unigrams
            for word in words:
                self.unigram_counts[word] += 1
            
            # Count n-grams
            for i in range(len(words) - self.n + 1):
                context = tuple(words[i:i + self.n - 1])
                word = words[i + self.n - 1]
                ngram_counts[context][word] += 1
                context_counts[context] += 1
        
        # Calculate probabilities with add-k smoothing
        k = 0.1  # Smoothing parameter
        vocab_size = len(self.vocabulary)
        
        for context, word_counts in ngram_counts.items():
            context_total = context_counts[context]
            for word, count in word_counts.items():
                # Add-k smoothing
                self.language_model[context][word] = \
                    (count + k) / (context_total + k * vocab_size)
        
        print(f"Language model trained on {len(corpus)} sentences")
    
    def get_language_model_prob(self, context, word):
        """
        Get probability of word given context from language model.
        """
        context = tuple(context)
        if context in self.language_model and word in self.language_model[context]:
            return self.language_model[context][word]
        else:
            # Backoff to unigram probability
            total_words = sum(self.unigram_counts.values())
            return (self.unigram_counts[word] + 0.1) / (total_words + 0.1 * len(self.vocabulary))
    
    def get_translation_prob(self, correct_word, misspelled_word):
        """
        Get P(misspelled|correct) from translation model.
        """
        if correct_word in self.translation_probs:
            if misspelled_word in self.translation_probs[correct_word]:
                return self.translation_probs[correct_word][misspelled_word]
        
        # Fallback: use edit distance-based probability
        edit_dist = self._edit_distance(correct_word, misspelled_word)
        if edit_dist == 0:
            return 1.0
        else:
            return 1.0 / (1.0 + edit_dist * 2)
    
    def generate_candidates(self, word, max_edit_distance=2):
        """
        Generate candidate corrections for a word based on edit distance.
        """
        candidates = set()
        
        # If word is already in vocabulary, it's a candidate
        if word.lower() in self.vocabulary:
            candidates.add(word.lower())
        
        # Generate words within edit distance
        for vocab_word in self.vocabulary:
            if vocab_word in ['<s>', '</s>']:
                continue
            if self._edit_distance(word.lower(), vocab_word) <= max_edit_distance:
                candidates.add(vocab_word)
        
        return candidates if candidates else {word.lower()}
    
    def should_correct(self, word):
        """
        Decide if a word should be considered for correction.
        
        Args:
            word: Word to check
        
        Returns:
            bool: True if word should be corrected, False otherwise
        """
        word_lower = word.lower()
        
        # Don't correct if word is in vocabulary
        if word_lower in self.vocabulary and word_lower not in ['<s>', '</s>']:
            return False
        
        # Don't correct very short words (likely correct or intentional)
        if len(word) <= 2:
            return False
        
        # Don't correct proper nouns (capitalized words)
        if word[0].isupper() and len(word) > 1 and word[1:].islower():
            return False
        
        # Don't correct likely acronyms (all caps, short)
        if word.isupper() and len(word) <= 5:
            return False
        
        # Don't correct words with numbers (likely codes, IDs, etc.)
        if any(char.isdigit() for char in word):
            return False
        
        # Don't correct words with special characters (emails, URLs, etc.)
        if any(char in '@._-/' for char in word):
            return False
        
        return True
    
    def correct_word(self, word, context, confidence_threshold=0.7, max_edit_distance=3):
        """
        Correct a single word using SMT approach with safety guardrails.
        
        Args:
            word: Word to correct
            context: List of previous words (for language model)
            confidence_threshold: Minimum confidence to apply correction (0-1)
            max_edit_distance: Maximum allowed edit distance for correction
        
        Returns:
            Best correction for the word
        """
        word_lower = word.lower()
        
        # Guardrail 0: Check if word should be corrected at all
        if not self.should_correct(word):
            return word_lower
        
        # Guardrail 1: If word is already in vocabulary, don't change it
        if word_lower in self.vocabulary and word_lower not in ['<s>', '</s>']:
            return word_lower
        
        candidates = self.generate_candidates(word)
        
        if not candidates:
            return word_lower
        
        # Include original word as a candidate
        candidates.add(word_lower)
        
        best_word = word_lower
        best_score = float('-inf')
        original_score = float('-inf')
        
        for candidate in candidates:
            # P(misspelled|correct) - translation model
            trans_prob = self.get_translation_prob(candidate, word_lower)
            
            # P(correct|context) - language model
            lm_prob = self.get_language_model_prob(context, candidate)
            
            # Combined score (log probabilities to avoid underflow)
            score = math.log(trans_prob + 1e-10) + math.log(lm_prob + 1e-10)
            
            if candidate == word_lower:
                original_score = score
            
            if score > best_score:
                best_score = score
                best_word = candidate
        
        # Guardrail 2: Check edit distance - don't change if correction is too different
        edit_dist = self._edit_distance(word_lower, best_word)
        if edit_dist > max_edit_distance:
            return word_lower
        
        # Guardrail 3: Confidence threshold - only change if significantly better
        if original_score != float('-inf'):
            confidence = best_score - original_score
            if confidence < math.log(confidence_threshold):
                return word_lower
        
        # Guardrail 4: If the best candidate is not in vocabulary and original is short, keep original
        if best_word not in self.vocabulary and len(word_lower) <= 3:
            return word_lower
        
        return best_word
    
    def correct_sentence(self, sentence, confidence_threshold=0.7, max_edit_distance=3):
        """
        Correct spelling errors in a sentence with safety guardrails.
        
        Args:
            sentence: Input sentence with potential spelling errors
            confidence_threshold: Minimum confidence to apply correction (0-1)
                                Higher = more conservative (0.5-0.9 recommended)
            max_edit_distance: Maximum allowed edit distance for correction (2-4 recommended)
        
        Returns:
            Corrected sentence
        """
        words = sentence.split()
        corrected_words = []
        context = ['<s>'] * (self.n - 1)
        
        for word in words:
            # Preserve punctuation
            match = re.match(r"(\w+)(.*)", word)
            if match:
                word_part = match.group(1)
                punct_part = match.group(2)
                
                corrected_word = self.correct_word(
                    word_part, 
                    context[-(self.n-1):],
                    confidence_threshold=confidence_threshold,
                    max_edit_distance=max_edit_distance
                )
                corrected_words.append(corrected_word + punct_part)
                
                # Update context
                context.append(corrected_word)
            else:
                corrected_words.append(word)
        
        return ' '.join(corrected_words)

## Statistical spelling corrector - utility functions

In [2]:
def load_dataset(parallel_file, lm_file):
    """
    Load training datasets for use in Statistical spelling corrector.
    
    Args:
        parallel_file: Path to parallel corpus file
        lm_file: Path to language model corpus file
    
    Returns:
        parallel_corpus, lm_corpus
    """
    # Load parallel corpus
    parallel_corpus = []
    with open(parallel_file, 'r', encoding='utf-8') as f:
        for line in f:
            if '\t' in line:
                misspelled, correct = line.strip().split('\t')
                parallel_corpus.append((misspelled, correct))
    
    # Load LM corpus
    lm_corpus = []
    with open(lm_file, 'r', encoding='utf-8') as f:
        for line in f:
            sentence = line.strip()
            if sentence:
                lm_corpus.append(sentence)
    
    return parallel_corpus, lm_corpus

def load_test_data(test_file):
    """
    Load test data from file containing paragraphs with misspellings.

    Args:
        test_file: Path to test data file with paragraphs separated by blank lines
    
    Returns:
        List of test paragraphs
    """
    test_paragraphs = []
    current_paragraph = []
    
    with open(test_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            # Empty line indicates end of paragraph
            if not line:
                if current_paragraph:
                    # Join the lines to form a complete paragraph
                    paragraph_text = ' '.join(current_paragraph)
                    test_paragraphs.append(paragraph_text)
                    current_paragraph = []
            else:
                # Add line to current paragraph (it's continuous text)
                current_paragraph.append(line)
        
        # Add the last paragraph if exists
        if current_paragraph:
            paragraph_text = ' '.join(current_paragraph)
            test_paragraphs.append(paragraph_text)
    
    return test_paragraphs


def main():
    """
    Main function demonstrating the Statistical spelling corrector with guardrails.
    """
    print("=" * 60)
    print("Statistics-based Spelling Corrector for English")
    print("=" * 60)
    print()
    
    # Load data
    parallel_corpus, lm_corpus = load_dataset(
        "/kaggle/input/statistical-spelling-correction-training-data/parallel_corpus.txt", 
        "/kaggle/input/statistical-spelling-correction-training-data/lm_corpus.txt"
    )
    
    # Initialize and train the corrector
    corrector = StatisticalSpellingCorrector(n=2)
    
    # Train translation model
    corrector.train_translation_model(parallel_corpus)
    print()
    
    # Train language model
    corrector.train_language_model(lm_corpus)
    print()
    
    # Test paragraphs with incorrect spellings
    print("\n" + "=" * 60)
    print("Testing:")
    print("=" * 60)
    test_data_file = "/kaggle/input/test-dataset-english/test_paragraphs-incorrect_spellings.txt"
    
    try:
        test_paragraphs = load_test_data(test_data_file)
        print(f"Loaded {len(test_paragraphs)} test paragraphs")
        print()
        total_corrections = 0
        
        # Test paragraphs
        for i, paragraph in enumerate(test_paragraphs, 1):
            print(f"\n--- Test Paragraph {i} ---")
            print(f"\nOriginal Paragraph:\n{paragraph}")
            
            corrected = corrector.correct_sentence(
                paragraph,
                confidence_threshold=0.7,
                max_edit_distance=3
            )
            
            print(f"\nCorrected Paragraph:\n{corrected}")
            
            # Count corrections made
            original_words = paragraph.lower().split()
            corrected_words = corrected.lower().split()
            corrections = sum(1 for o, c in zip(original_words, corrected_words) if o != c)
            print(f"\nCorrections made: {corrections} words")
            total_corrections += corrections

        print("\n" + "=" * 60)
        print(f"Total test paragraphs: {len(test_paragraphs)}")
        print(f"Total corrections made: {total_corrections} words")
        print("=" * 60)
    
    except FileNotFoundError:
        print(f"\nNote: {test_data_file} not found")
        print("Skipping advanced vocabulary test")


## Testing statistical spelling corrector on its own

In [3]:
main()

Statistics-based Spelling Corrector for English

Training translation model...
Translation model trained with 3764 vocabulary words

Training 2-gram language model...
Language model trained on 2704 sentences


Testing:
Loaded 10 test paragraphs


--- Test Paragraph 1 ---

Original Paragraph:
The meticulus cartografical analisys and extensiv logistical preparatons for the expedtion were deamed absolutly indespensible for its succes. Travercing the formidabel, glacier-carved valeys and precipitous ridglines of the unchartd mountain range presentd not mearly a phisical trial of endurence, but a profund psycological crucibel for the entier team. Each memeber undersood that a singel miscalculation in provissions, aclimatization scheduals, or navigasional stratogy could precipatate a catastrophic faliure, rendring their ambicious undertaking a cautionary tale rather than a triumfant chronicel of discovery.

Corrected Paragraph:
the meticulous cartographical analysis and extensive logistical 

## Statistical spelling corrector object for pipeline usage

In [4]:
print("=" * 60)
print("Statistics-based Spelling Corrector for English")
print("=" * 60)
print()

# Load data
parallel_corpus, lm_corpus = load_dataset(
    "/kaggle/input/statistical-spelling-correction-training-data/parallel_corpus.txt", 
    "/kaggle/input/statistical-spelling-correction-training-data/lm_corpus.txt"
)

# Initialize and train the corrector
print("Initializing corrector object...")
spelling_corrector = StatisticalSpellingCorrector(n=2)

# Train translation model
print("Training the translation model...")
spelling_corrector.train_translation_model(parallel_corpus)
print()

# Train language model
print("Training the language model...")
spelling_corrector.train_language_model(lm_corpus)
print()

print("Spelling corrector object successfully loaded and ready to use!")

Statistics-based Spelling Corrector for English

Initializing corrector object...
Training the translation model...
Training translation model...
Translation model trained with 3764 vocabulary words

Training the language model...
Training 2-gram language model...
Language model trained on 2704 sentences

Spelling corrector object successfully loaded and ready to use!


## Example usage

In [5]:
spelling_corrector.correct_sentence(
    "the proffesor gave an exellent lexture on environmental conservation and how it protects precious ecosystems for future generations.",
    confidence_threshold=0.7,   # optional
    max_edit_distance=3         # optional
)

'the professor gave an excellent lecture on environmental conservation and how it protects precious ecosystems for future generations.'

# Stage 2: Intermediate complex english to simple english converter

## Set environment

In [6]:
stage_2_finetune = False

In [7]:
%%capture
!pip install contractions
!pip install unsloth==2025.10.1
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

## Utility functions

In [ ]:
import contractions
import pandas as pd
from IPython.display import display
from huggingface_hub import HfApi, login, snapshot_download

def normalize_text(text):
    """
    Expands contractions and custom slang based on predefined lists.
    """
    # Expand standard contractions (e.g., "don't" to "do not")
    text = contractions.fix(text, slang=True)

    return text

import re
def paragraph_to_sentences(paragraph_string):
  sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z"])|(?<=[.!?])\s*$', paragraph_string.strip())
  return [s.strip() for s in sentences if s.strip()]


# --- Configuration ---
HF_USER = "Eranda-sathsara"
STAGE2_LOCAL_DIR = "gemma3-finetuned"
HF_REPO_ID_2 = f"{HF_USER}/{STAGE2_LOCAL_DIR}"
HF_TOKEN = "<TOKEN>"


def upload_to_huggingface_2():
    """Upload minimized model to Hugging Face Hub."""
    try:
        print("🔑 Logging into Hugging Face...")
        login(token=HF_TOKEN)
        api = HfApi()
        api.create_repo(repo_id=HF_REPO_ID_2, private=False, exist_ok=True, token=HF_TOKEN)
        api.upload_folder(
            folder_path=STAGE2_LOCAL_DIR,
            repo_id=HF_REPO_ID_2,
            repo_type="model",
            commit_message="Upload gemma3 finetuned",
            token=HF_TOKEN,
        )
        print(f"✅ Uploaded to: https://huggingface.co/{HF_REPO_ID_2}")
    except Exception as e:
        print(f"⚠️ Upload failed: {e}")

## Finetune

In [9]:
from unsloth import FastModel
import torch
max_seq_length = 2048

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-10-28 13:03:18.228187: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761656598.578344      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761656598.687268      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [10]:
if stage_2_finetune:
    model, tokenizer = FastModel.from_pretrained(
        model_name = "unsloth/gemma-3-270m-it-unsloth-bnb-4bit",
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        load_in_8bit = False,
        full_finetuning = False,
    )
    
    model = FastModel.get_peft_model(
        model,
        r = 128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 256,
        lora_dropout = 0,
        bias = "none",   
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False, 
        loftq_config = None,
    )
    
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "gemma3",
    )
    
    from datasets import Dataset
    import pandas as pd
    df1 = pd.read_csv("/kaggle/input/test-file/comp-simp1.csv")
    df2 = pd.read_csv("/kaggle/input/test-file/comp-simp2.csv")
    df = pd.concat([df1,df2])
    df = df[['input', 'output']]
    dataset = Dataset.from_pandas(df)
    
    display(dataset)
    
    def convert_to_chatml(example):
        return {
            "conversations": [
                {"role": "system", "content": "Simplify the sentence while retaining all the information and meaning"},
                {"role": "user", "content": example["input"]},
                {"role": "assistant", "content": example["output"]}
            ]
        }
    
    dataset = dataset.map(
        convert_to_chatml
    )
    
    def formatting_prompts_func(examples):
       convos = examples["conversations"]
       texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
       return { "text" : texts, }
    
    dataset = dataset.map(formatting_prompts_func, batched = True)
    
    # split the datsset into train and test
    dataset = dataset.train_test_split(test_size=0.2)
    display(dataset)

In [11]:
from trl import SFTTrainer, SFTConfig

if stage_2_finetune:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset['train'],
        eval_dataset=dataset['test'],  
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=32,       
            gradient_accumulation_steps=10,        
            warmup_steps=20,
            num_train_epochs=10,                   
            learning_rate=5e-5,                   
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",                     
            eval_strategy="epoch",       
            save_strategy="epoch",             
            load_best_model_at_end=True,       
            metric_for_best_model="loss",      
            greater_is_better=False,           
            fp16=True,                     
            ddp_find_unused_parameters=False,
        ),
    )
    
    
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<start_of_turn>user\n",
        response_part = "<start_of_turn>model\n",
    )
    
    trainer_stats = trainer.train()
    
    # training stats
    display(trainer_stats.metrics)
    
    model.save_pretrained(STAGE2_LOCAL_DIR)  # Local saving
    tokenizer.save_pretrained(STAGE2_LOCAL_DIR)

    upload_to_huggingface_2()
    

    

## Inference

In [12]:
from unsloth import FastLanguageModel
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name = HF_REPO_ID_2 , # YOUR TRAINED MODEL
    max_seq_length = 2048,
    load_in_4bit = False,
)
FastLanguageModel.for_inference(model2)

==((====))==  Unsloth 2025.10.1: Fast Gemma3 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForCausalLM(
      (model): Gemma3TextModel(
        (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x Gemma3DecoderLayer(
            (self_attn): Gemma3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=640, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=640, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [13]:
def convert_lang_eng(para):
    """
    the main function that can be used for the final system
    uses the fine tuned model note that for this function to be used 
    1st a model should be fine tuned or have loaded a fine tuned model
    input: the paragraph that needs to be simplified
    output: simplified paragraph
    """
    
    sa = paragraph_to_sentences(para)
    ss_simp = ""
    for s in sa:
        messages = [
            {'role': 'system','content':"Simplify the sentence while retaining all the information and meaning"},
            {"role" : 'user', 'content' : s}
        ]
        input_text = tokenizer2.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer2(input_text, return_tensors="pt").to("cuda")
        
        # Get the length of the input tokens
        input_ids_len = inputs["input_ids"].shape[1]
        
        # Generate the response
        outputs = model2.generate(**inputs, max_new_tokens=128, use_cache=True,temperature = 0.2, top_p = 0.95, top_k = 64,)
        
        # Slice the output tensor to get only the generated tokens
        new_tokens = outputs[0, input_ids_len:]
        
        # Decode only the new tokens to get the clean response
        clean_response = tokenizer2.decode(new_tokens, skip_special_tokens=True)
        
        
        # print("Prompt:",s)
        clean_response = normalize_text(clean_response)
        # print("Generated Response:", clean_response)
        ss_simp += clean_response
        ss_simp += " "
    # print(ss_simp)

    return ss_simp

In [14]:
#example usage illustration
para = "Subsequent to the initial astronomical observations, a more rigorous spectroscopic analysis was undertaken to deconstruct the constituent elements responsible for the scintillating, ethereal luminescence of the aurora borealis. The phenomenon, long a subject of folklore and mythological interpretation, is now understood as the magnificent confluence of charged solar particulates with the Earth's magnetosphere. Nevertheless, this empirical demystification does nothing to diminish the profound, almost transcendental, sensation of awe that invariably captivates any individual fortunate enough to witness the celestial ballet firsthand."
convert_lang_eng(para)

"After the initial big photos, a more precise analysis was done to break down the elements that make the Aurora. The myth, long-loved and explained by stories, is now seen as the big mix of charged solar particles with the Earth's magnet. Despite this simple explanation, it does not make the feeling of awe that one might feel seeing the stars. "

# Stage 3: Final level neural translation

## Quantized MBART Translation Pipeline

In [ ]:
import os
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, BitsAndBytesConfig
from huggingface_hub import HfApi, login, snapshot_download

# --- Configuration ---
HF_USER = "Eranda-sathsara"
HF_REPO_ID = f"{HF_USER}/mbart-large-50-en-si-finetuned"
BASE_MODEL = "facebook/mbart-large-50-many-to-many-mmt"
LOCAL_DIR = "mbart_en_si_finetuned_local_minimized"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HF_TOKEN = "<TOKEN>"


def load_or_minimize_model():
    """Load minimized fine-tuned model if available, else minimize and save base model."""
    
    # 1️⃣ Load minimized model if already saved
    if os.path.exists(LOCAL_DIR):
        print(f"✅ Found local minimized model at {LOCAL_DIR}")
        model = MBartForConditionalGeneration.from_pretrained(LOCAL_DIR).to(DEVICE)
        tokenizer = MBart50TokenizerFast.from_pretrained(LOCAL_DIR)
        return model, tokenizer

    # 2️⃣ Try downloading fine-tuned model from Hugging Face (if already uploaded)
    try:
        print(f"🔍 Checking for fine-tuned model on Hugging Face: {HF_REPO_ID}")
        snapshot_download(repo_id=HF_REPO_ID, local_dir=LOCAL_DIR, token=HF_TOKEN)
        model = MBartForConditionalGeneration.from_pretrained(LOCAL_DIR).to(DEVICE)
        tokenizer = MBart50TokenizerFast.from_pretrained(LOCAL_DIR)
        print("✅ Loaded fine-tuned model from Hub.")
        return model, tokenizer
    except Exception as e:
        print(f"⚠️ Fine-tuned model not found ({e}). Proceeding with base model...")

    # 3️⃣ Download and minimize base model
    print(f"🚀 Downloading base model: {BASE_MODEL}")
    base_dir = snapshot_download(repo_id=BASE_MODEL, token=HF_TOKEN)

    # ✅ Configure 8-bit quantization to minimize memory & disk space
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = MBartForConditionalGeneration.from_pretrained(
        base_dir,
        quantization_config=bnb_config,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    tokenizer = MBart50TokenizerFast.from_pretrained(base_dir)
    tokenizer.src_lang = "en_XX"
    tokenizer.tgt_lang = "si_LK"

    print("✅ Base model minimized using 8-bit quantization.")
    print("💾 Saving minimized model locally...")
    os.makedirs(LOCAL_DIR, exist_ok=True)
    model.save_pretrained(LOCAL_DIR)
    tokenizer.save_pretrained(LOCAL_DIR)
    print(f"✅ Minimized model saved to {LOCAL_DIR}")

    return model, tokenizer


def translate_en_to_si(model, tokenizer, texts):
    """Translate English text(s) to Sinhala using minimized or fine-tuned model."""
    if isinstance(texts, str):
        texts = [texts]
    tokenizer.src_lang = "en_XX"
    tokenizer.tgt_lang = "si_LK"

    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[tokenizer.tgt_lang],
            max_length=500,
            num_beams=4,
        )
    return [tokenizer.decode(g, skip_special_tokens=True) for g in generated]


def upload_to_huggingface():
    """Upload minimized model to Hugging Face Hub."""
    try:
        print("🔑 Logging into Hugging Face...")
        login(token=HF_TOKEN)
        api = HfApi()
        api.create_repo(repo_id=HF_REPO_ID, private=False, exist_ok=True, token=HF_TOKEN)
        api.upload_folder(
            folder_path=LOCAL_DIR,
            repo_id=HF_REPO_ID,
            repo_type="model",
            commit_message="Upload minimized English→Sinhala MBART model (8-bit)",
            token=HF_TOKEN,
        )
        print(f"✅ Uploaded to: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"⚠️ Upload failed: {e}")


if __name__ == "__main__":
    model3, tokenizer3 = load_or_minimize_model()

    print("\n--- English → Sinhala Translation Test ---")
    samples = [
        "What is your name?",
        "Good morning, how are you today?",
    ]
    for src, tgt in zip(samples, translate_en_to_si(model3, tokenizer3, samples)):
        print(f"{src} → {tgt}\n")

    # Optional: upload minimized version
    # upload_to_huggingface()


🔍 Checking for fine-tuned model on Hugging Face: Eranda-sathsara/mbart-large-50-en-si-finetuned


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/992 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

✅ Loaded fine-tuned model from Hub.

--- English → Sinhala Translation Test ---
What is your name? → මොකක්ද ඔයාගේ නම?

Good morning, how are you today? → සුබ උදෑසනක්, කොහොමද අද?



# Full System

## Final stage functions

In [16]:
def stage1(s: str) -> str:
    return spelling_corrector.correct_sentence(s, confidence_threshold=0.7, max_edit_distance=3)

def stage2(s: str) -> str: 
    s = convert_lang_eng(s)
    return s

def stage3(s: str) -> str:
    return translate_en_to_si(model3, tokenizer3, [s])[0]

## Gradio

In [17]:
!pip install gradio --quiet
import gradio as gr

def process_translation(eng_text):
    """
    Process text through all three functions sequentially
    Returns all intermediate steps for display
    """
    # Step 1: Spelling correction
    corrected_text = stage1(eng_text)
    
    # Step 2: Simplification
    simplified_text = stage2(corrected_text)
    
    # Step 3: Translation
    sinhala_text = stage3(simplified_text)
    
    return sinhala_text, corrected_text, simplified_text

def translate_pipeline(text):
    """
    Main function called by the UI
    """
    if not text.strip():
        return "Please enter some text to translate.", "", "", "", ""
    
    try:
        sinhala, corrected, simplified = process_translation(text)
        return sinhala, corrected, simplified, text, sinhala
    except Exception as e:
        return f"Error: {str(e)}", "", "", "", ""

# Create the Gradio interface
with gr.Blocks(title="English to Sinhala Translator") as interface:
    gr.Markdown("# 🌐 English → Sinhala Translation Pipeline")
    gr.Markdown("Enter English text below. The system will correct spellings, simplify the text, and translate to Sinhala.")
    
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="English Input",
                placeholder="Type or paste your English text here...",
                lines=5
            )
            translate_btn = gr.Button("🔄 Translate", variant="primary", size="lg")
    
    with gr.Row():
        with gr.Column():
            output_text = gr.Textbox(
                label="📝 Sinhala Translation",
                lines=5,
                interactive=False
            )
    
    # Collapsible sections for intermediate steps
    with gr.Accordion("🔍 View Intermediate Steps", open=False):
        gr.Markdown("### Processing Pipeline Details")

        with gr.Row():
            with gr.Column():
                english_text = gr.Textbox(
                    label="Step 0: Original English Input",
                    lines=5,
                    interactive=False
                )
                
        with gr.Row():
            with gr.Column():
                corrected_text = gr.Textbox(
                    label="Step 1: Spelling Correction",
                    lines=5,
                    interactive=False
                )
        
        with gr.Row():
            with gr.Column():
                simplified_text = gr.Textbox(
                    label="Step 2: Text Simplification",
                    lines=5,
                    interactive=False
                )

        with gr.Row():
            with gr.Column():
                sinhala_text = gr.Textbox(
                    label="Step 3: Final Sinhala Output",
                    lines=5,
                    interactive=False
                )
    
    # Connect the button to the function
    translate_btn.click(
        fn=translate_pipeline,
        inputs=[input_text],
        outputs=[output_text, corrected_text, simplified_text, english_text, sinhala_text]
    )
    
    # Optional: Add examples
    gr.Examples(
        examples=[
            ["The proffesor gave an exellent lexture on environmental conservation and how it protects precious ecosystems for future generations."],
            ["The sun rizes early in the eastern sky, and the ancient ruins wispered tales of a forgoten civilization. "],
        ],
        inputs=input_text
    )

interface.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://bb95ac5a78393d3265.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
